In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler


In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D3 
clinical_train.isnull().sum().sum()

0

### COX assumption in Train data

In [6]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.97      0.05
LBP_003_PET                                                      0.00 0.97      0.05
LBP_012_CT                                                       0.00 0.96      0.06
LBP_012_PET                                                      0.01 0.94      0.09
LBP_021_CT                                                       0.00 0.98      0.04
LBP_021_PET                                                      0.01 0.94      0.10
LBP_030_CT                                                       0.00 0.99      0.02
LBP_030_PET       

In [7]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [8]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.95      0.07
LBP_003_PET                                                      0.08 0.77      0.37
LBP_012_CT                                                       0.00 0.95      0.07
LBP_012_PET                                                      0.18 0.67      0.57
LBP_021_CT                                                       0.15 0.69      0.53
LBP_021_PET                                                      0.01 0.94      0.09
LBP_030_CT                                                       0.00 0.99      0.01
LBP_030_PET       

In [9]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


## Test dataset: MAASTRO 

In [10]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [11]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [12]:
# Need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [13]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [14]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [15]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [16]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [17]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [18]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [19]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [20]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = RobustScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [23]:
# Divide the X_MAASTRO into numerical part and categorical part 
X_MAASTRO_categorical = X_MAASTRO[categorical_columns]
X_MAASTRO_numeric = X_MAASTRO.drop(categorical_columns, axis=1)

# Save the column name and index of the numeric part
X_MAASTRO_numeric_columns = X_MAASTRO_numeric.columns
X_MAASTRO_numeric_index = X_MAASTRO_numeric.index

In [24]:
# Standardize the numeric part 
X_MAASTRO_numeric_std = scaler.transform(X_MAASTRO_numeric)

# Change the standardized part into a dataframe 
X_MAASTRO_numeric_std = pd.DataFrame(X_MAASTRO_numeric_std, columns=X_MAASTRO_numeric_columns, index=X_MAASTRO_numeric_index)

# Concat the standardized part with the categorical part 
X_MAASTRO_std = pd.concat([X_MAASTRO_categorical, X_MAASTRO_numeric_std], axis=1)

# Change the column order of X_MAASTRO_std
X_MAASTRO_std = X_MAASTRO_std[original_X.columns]

In [25]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [26]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 13:47:12,102] A new study created in memory with name: no-name-71d5a103-aac7-4a28-85ba-6bbd324c38a4


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-17 13:47:12,390] Trial 0 failed with parameters: {} because of the following error: ValueError('search direction contains NaN or infinite values').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_1413/4166082094.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 454, in fit
    raise ValueError("search direction contains NaN or infinite values")
ValueError: search direction contains NaN or infinite values
[W 2024-04-17 13:47:12,393] Trial 0 failed with value None.


ValueError: search direction contains NaN or infinite values

In [27]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

In [28]:
# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [29]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [30]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [31]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

ValueError: search direction contains NaN or infinite values

In [34]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [35]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [36]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:50:43,355] A new study created in memory with name: no-name-6a676896-0cf9-40cf-99de-c18541662832


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7566964285714286
Fold 3 C-index: 0.6372549019607843


[I 2024-04-17 13:50:43,840] A new study created in memory with name: no-name-d9219d17-9278-4a77-a309-e3ee5f2d2c55


Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.5774647887323944
[I 2024-04-17 13:50:43,834] Trial 0 finished with value: 0.6505821991392386 and parameters: {}. Best is trial 0 with value: 0.6505821991392386.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6505821991392386], datetime_start=datetime.datetime(2024, 4, 17, 13, 50, 43, 389242), datetime_complete=datetime.datetime(2024, 4, 17, 13, 50, 43, 834553), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6505821991392386


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651830695444
Fold 2 IBS: 0.22157791625059692
Fold 3 IBS: 0.20453594280733517
Fold 4 IBS: 0.22473802997425457
Fold 5 IBS: 0.21812431597978285
[I 2024-04-17 13:50:44,270] Trial 0 finished with value: 0.21659054466378475 and parameters: {}. Best is trial 0 with value: 0.21659054466378475.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054466378475], datetime_start=datetime.datetime(2024, 4, 17, 13, 50, 43, 880608), datetime_complete=datetime.datetime(2024, 4, 17, 13, 50, 44, 270696), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054466378475


In [37]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [38]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.651
train_ibs:  0.217


#### Test

In [39]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [40]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.592


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [41]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [42]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:50:44,947] A new study created in memory with name: no-name-087afed9-a03d-490f-9279-007bc35aea46


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.8235294117647058


[I 2024-04-17 13:50:45,854] A new study created in memory with name: no-name-1aa0a6ba-a2e4-49fc-b662-43f97cb57627


Fold 4 C-index: 0.6455696202531646
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:50:45,820] Trial 0 finished with value: 0.706327046815744 and parameters: {}. Best is trial 0 with value: 0.706327046815744.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.706327046815744], datetime_start=datetime.datetime(2024, 4, 17, 13, 50, 44, 980220), datetime_complete=datetime.datetime(2024, 4, 17, 13, 50, 45, 820067), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.706327046815744


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21388429370731754
Fold 2 IBS: 0.1585758545159384
Fold 3 IBS: 0.15832262210958478
Fold 4 IBS: 0.27197836239155776
Fold 5 IBS: 0.20706497029738247
[I 2024-04-17 13:50:46,647] Trial 0 finished with value: 0.20196522060435615 and parameters: {}. Best is trial 0 with value: 0.20196522060435615.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20196522060435615], datetime_start=datetime.datetime(2024, 4, 17, 13, 50, 45, 875533), datetime_complete=datetime.datetime(2024, 4, 17, 13, 50, 46, 647739), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20196522060435615


In [43]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [44]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.706
train_ibs:  0.202


#### Test

In [45]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [46]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.597


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.221


In [47]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [48]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:50:46,815] A new study created in memory with name: no-name-da201069-1f85-4466-aae8-dc79d4d843e6


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7232142857142857
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:50:47,704] Trial 0 finished with value: 0.7117757136242396 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7117757136242396.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7511737089201878
[I 2024-04-17 13:50:48,488] Trial 1 finished with value: 0.7153028518819318 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7153028518819318.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.755868544600939
[I 2024-04-17 13:50:49,206] Trial 2 finished with value: 0.7125442957049593 and parameters: {'l1_ratio': 0.22692876841884668}. B

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.755868544600939
[I 2024-04-17 13:51:03,817] Trial 24 finished with value: 0.717085700874622 and parameters: {'l1_ratio': 0.3525262253351147}. Best is trial 23 with value: 0.717085700874622.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.7511737089201878
[I 2024-04-17 13:51:04,466] Trial 25 finished with value: 0.7144589700253915 and parameters: {'l1_ratio': 0.32881905267396466}. Best is trial 23 with value: 0.717085700874622.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.755868544600939
[I 2024-04-17 13:51:05,111] Trial 26 finished with value: 0.7125442957049593 and parameters: {'l1_ratio': 0.22313754734009067}. Bes

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.7511737089201878
[I 2024-04-17 13:51:19,328] Trial 48 finished with value: 0.7144589700253915 and parameters: {'l1_ratio': 0.33659813406040173}. Best is trial 23 with value: 0.717085700874622.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7370892018779343
[I 2024-04-17 13:51:20,018] Trial 49 finished with value: 0.6981197926179338 and parameters: {'l1_ratio': 0.05663011584206612}. Best is trial 23 with value: 0.717085700874622.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.755868544600939
[I 2024-04-17 13:51:20,740] Trial 50 finished with value: 0.7152124515749021 and parameters: {'l1_ratio': 0.4358244534444967}. 

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.21388436919013662
Fold 2 IBS: 0.15916880039525647
Fold 3 IBS: 0.1608534236085006
Fold 4 IBS: 0.24119149140634347
Fold 5 IBS: 0.2068242913152905
[I 2024-04-17 13:51:55,308] Trial 0 finished with value: 0.1963844751831055 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.1963844751831055.
Fold 1 IBS: 0.2138847246005204
Fold 2 IBS: 0.16260871942873492
Fold 3 IBS: 0.1647087664941015
Fold 4 IBS: 0.2290336517349254
Fold 5 IBS: 0.2077575455483112
[I 2024-04-17 13:51:56,082] Trial 1 finished with value: 0.19559868156131868 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.19559868156131868.
Fold 1 IBS: 0.21388488162509872
Fold 2 IBS: 0.16433562477400576
Fold 3 IBS: 0.16576586117353376
Fold 4 IBS: 0.224940465607407
Fold 5 IBS: 0.2082372732694083
[I 2024-04-17 13:51:56,832] Trial 2 finished with value: 0.1954328212898907 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.1954328212898907.
Fol

Fold 1 IBS: 0.21388664449810943
Fold 2 IBS: 0.17415098459380943
Fold 3 IBS: 0.1710197787221419
Fold 4 IBS: 0.2077756372328316
Fold 5 IBS: 0.21106836087702271
[I 2024-04-17 13:52:16,769] Trial 25 finished with value: 0.19558028118478302 and parameters: {'l1_ratio': 0.06756100018912164}. Best is trial 16 with value: 0.19476258456850729.
Fold 1 IBS: 0.21388461106227552
Fold 2 IBS: 0.16136483931571033
Fold 3 IBS: 0.16369840206833908
Fold 4 IBS: 0.23142567200158498
Fold 5 IBS: 0.20740945596574717
[I 2024-04-17 13:52:17,933] Trial 26 finished with value: 0.1955565960827314 and parameters: {'l1_ratio': 0.3527001258697712}. Best is trial 16 with value: 0.19476258456850729.
Fold 1 IBS: 0.21388527447177116
Fold 2 IBS: 0.16758270103472897
Fold 3 IBS: 0.167757772915696
Fold 4 IBS: 0.21580089819830942
Fold 5 IBS: 0.2092761998158501
[I 2024-04-17 13:52:18,901] Trial 27 finished with value: 0.1948605692872711 and parameters: {'l1_ratio': 0.14926618171209116}. Best is trial 16 with value: 0.1947625845

Fold 1 IBS: 0.21388481103829365
Fold 2 IBS: 0.1635776383794378
Fold 3 IBS: 0.16534058181800254
Fold 4 IBS: 0.2268193955278957
Fold 5 IBS: 0.20804684108284635
[I 2024-04-17 13:52:43,796] Trial 50 finished with value: 0.19553385356929517 and parameters: {'l1_ratio': 0.2502435094853836}. Best is trial 16 with value: 0.19476258456850729.
Fold 1 IBS: 0.21388558340209102
Fold 2 IBS: 0.16950659347353192
Fold 3 IBS: 0.16872663625043585
Fold 4 IBS: 0.21193990109863584
Fold 5 IBS: 0.20974120742218041
[I 2024-04-17 13:52:44,563] Trial 51 finished with value: 0.194759984329375 and parameters: {'l1_ratio': 0.11748326911570342}. Best is trial 51 with value: 0.194759984329375.
Fold 1 IBS: 0.21388527732510756
Fold 2 IBS: 0.16760320538408446
Fold 3 IBS: 0.16776826690118662
Fold 4 IBS: 0.21575459111337
Fold 5 IBS: 0.20928166030074472
[I 2024-04-17 13:52:45,293] Trial 52 finished with value: 0.19485860020489867 and parameters: {'l1_ratio': 0.1488949868435126}. Best is trial 51 with value: 0.1947599843293

Fold 2 IBS: 0.16239750342334874
Fold 3 IBS: 0.16456201974352128
Fold 4 IBS: 0.22945154984812816
Fold 5 IBS: 0.20770967185975572
[I 2024-04-17 13:53:04,044] Trial 75 finished with value: 0.1956010904343784 and parameters: {'l1_ratio': 0.29468292253424555}. Best is trial 51 with value: 0.194759984329375.
Fold 1 IBS: 0.21388502233310006
Fold 2 IBS: 0.16566965231804723
Fold 3 IBS: 0.1665909613525107
Fold 4 IBS: 0.22097489860294398
Fold 5 IBS: 0.20867617295680735
[I 2024-04-17 13:53:04,920] Trial 76 finished with value: 0.19515934151268186 and parameters: {'l1_ratio': 0.19133788941063262}. Best is trial 51 with value: 0.194759984329375.
Fold 1 IBS: 0.21388517825050982
Fold 2 IBS: 0.16689172117757953
Fold 3 IBS: 0.16736821993035889
Fold 4 IBS: 0.21750519282022215
Fold 5 IBS: 0.20909360240651484
[I 2024-04-17 13:53:06,000] Trial 77 finished with value: 0.19494878291703704 and parameters: {'l1_ratio': 0.1629557902167686}. Best is trial 51 with value: 0.194759984329375.
Fold 1 IBS: 0.2138863962

In [49]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.717
train_ibs:  0.195


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.352181905423733)

test_cindex : 0.597


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.11748326911570342)

test_ibs:  0.221


In [53]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 13:53:38,519] A new study created in memory with name: no-name-c230e67a-c35f-40d1-ab50-98ec5e49a278


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.7383966244725738
Fold 5 C-index: 0.7230046948356808
[I 2024-04-17 13:54:04,068] Trial 0 finished with value: 0.7366032203169602 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7366032203169602.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.6854460093896714
[I 2024-04-17 13:54:09,407] Trial 1 finished with value: 0.7181620567966567 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 1 C-index: 0.6038961038961039
Fold 2 C-index: 0.7165178571428571
Fold 3 C-index: 0.7009803921568627
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6032863849765259
[I 2024-04-17 13:56:18,177] Trial 16 finished with value: 0.6633327721070438 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': None, 'min_weight_fraction_leaf': 0.1048543527585735, 'warm_start': False}. Best is trial 12 with value: 0.7438602547617209.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 13:56:26,605] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 417, 'oob_score': True, 'max_samples': 0.1424705672746025, 'max_features': None, 'min_weight_fraction_leaf': 0.20205510510599584, 'warm_start': F

Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.812206572769953
[I 2024-04-17 13:57:40,335] Trial 31 finished with value: 0.7936613975238874 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 215, 'oob_score': True, 'max_samples': 0.7258118850709611, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.020186907613437947, 'warm_start': True}. Best is trial 26 with value: 0.7952997362091728.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.8169014084507042
[I 2024-04-17 13:57:41,527] Trial 32 finished with value: 0.7892741864327053 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 8, 'n_estimators': 149, 'oob_score': True, 'max_samples': 0.7060378252364139, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0461866753469

...

Fold 1 IBS: 0.20444739845582136
Fold 2 IBS: 0.17446698341767913
Fold 3 IBS: 0.1910994493903993
Fold 4 IBS: 0.1668825886510828
Fold 5 IBS: 0.21103427734544067
[I 2024-04-17 14:11:13,031] Trial 32 finished with value: 0.18958613945208463 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 279, 'oob_score': False, 'max_samples': 0.6325115094238634, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0760629688237641}. Best is trial 21 with value: 0.18758259521522488.
Fold 1 IBS: 0.21394095082970305
Fold 2 IBS: 0.18957239420831615
Fold 3 IBS: 0.18237897227425182
Fold 4 IBS: 0.17846689832832333
Fold 5 IBS: 0.2185435449686763
[I 2024-04-17 14:11:14,024] Trial 33 finished with value: 0.19658055212185413 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 19, 'oob_score': False, 'max_samples': 0.7088178154422675, 'max_features': 'auto', 'min_weight_fraction_leaf':

Fold 1 IBS: 0.21397545241919333
Fold 2 IBS: 0.22060585341756236
Fold 3 IBS: 0.20519466411613427
Fold 4 IBS: 0.2248054198587066
Fold 5 IBS: 0.21758241895615993
[I 2024-04-17 14:14:06,034] Trial 48 finished with value: 0.2164327617535513 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 324, 'oob_score': True, 'max_samples': 0.1840472657347053, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.44962767788541247}. Best is trial 39 with value: 0.18741882102325463.
Fold 1 IBS: 0.20733215663048163
Fold 2 IBS: 0.19840873877308707
Fold 3 IBS: 0.1896647344947966
Fold 4 IBS: 0.2057146078926162
Fold 5 IBS: 0.2152447795126918
[I 2024-04-17 14:14:11,873] Trial 49 finished with value: 0.20327300346073468 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 20, 'max_depth': 4, 'n_estimators': 187, 'oob_score': True, 'max_samples': 0.4611544071310423, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.18697650055709675
Fold 2 IBS: 0.18964751614322145
Fold 3 IBS: 0.17157067951262514
Fold 4 IBS: 0.17048079186700552
Fold 5 IBS: 0.20812563312237994
[I 2024-04-17 14:21:57,264] Trial 64 finished with value: 0.18536022424046578 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 381, 'oob_score': True, 'max_samples': 0.36087148528744895, 'max_features': None, 'min_weight_fraction_leaf': 0.10501492814170305}. Best is trial 64 with value: 0.18536022424046578.
Fold 1 IBS: 0.185980269157479
Fold 2 IBS: 0.18929665176523391
Fold 3 IBS: 0.17261558628019577
Fold 4 IBS: 0.17528863593015434
Fold 5 IBS: 0.20987976574068176
[I 2024-04-17 14:22:33,945] Trial 65 finished with value: 0.18661218177474898 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 389, 'oob_score': False, 'max_samples': 0.35503340896452096, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.2139852535910293
Fold 2 IBS: 0.22079135086710738
Fold 3 IBS: 0.2049571432154721
Fold 4 IBS: 0.22449984942587378
Fold 5 IBS: 0.21784420818016068
[I 2024-04-17 14:27:43,525] Trial 80 finished with value: 0.21641556105592863 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 401, 'oob_score': False, 'max_samples': 0.23317876663764808, 'max_features': None, 'min_weight_fraction_leaf': 0.13195032778351135}. Best is trial 64 with value: 0.18536022424046578.
Fold 1 IBS: 0.18698963922132994
Fold 2 IBS: 0.19117908340974094
Fold 3 IBS: 0.1716425830420473
Fold 4 IBS: 0.1706227695034198
Fold 5 IBS: 0.20890166136319582
[I 2024-04-17 14:28:16,590] Trial 81 finished with value: 0.1858671473079468 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 7, 'min_samples_leaf': 3, 'max_depth': 3, 'n_estimators': 342, 'oob_score': False, 'max_samples': 0.3619448678095568, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.19987889711029083
Fold 2 IBS: 0.18568352389906653
Fold 3 IBS: 0.18380646487398802
Fold 4 IBS: 0.16033813976360883
Fold 5 IBS: 0.20642365546571148
[I 2024-04-17 14:37:30,050] Trial 96 finished with value: 0.18722613622253315 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 6, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 363, 'oob_score': False, 'max_samples': 0.4255182801365352, 'max_features': None, 'min_weight_fraction_leaf': 0.08056036420837291}. Best is trial 84 with value: 0.1832464308635407.
Fold 1 IBS: 0.19365837237754394
Fold 2 IBS: 0.183435405665406
Fold 3 IBS: 0.18008910470800357
Fold 4 IBS: 0.17339462295545943
Fold 5 IBS: 0.20781358095494984
[I 2024-04-17 14:37:59,432] Trial 97 finished with value: 0.18767821733227258 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 1, 'n_estimators': 344, 'oob_score': False, 'max_samples': 0.4693979321374093, 'max_features': None, 'min_weight_fraction_leaf': 0

In [55]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.864
train_ibs:  0.183


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [58]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=11, max_features='auto', max_leaf_nodes=15,
                     max_samples=0.9847467855272036, min_samples_leaf=4,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.04749646122417352,
                     n_estimators=230, random_state=123, warm_start=True)

test_cindex:  0.629


RandomSurvivalForest(max_depth=2, max_features=None, max_leaf_nodes=7,
                     max_samples=0.3902292236281065, min_samples_leaf=4,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.0959966989881746,
                     n_estimators=344, random_state=123)

test_ibs:  0.205


In [59]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [61]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 14:38:43,541] A new study created in memory with name: no-name-9fb45f90-7b94-470f-8bed-5b9c09a15c88


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.7605633802816901
[I 2024-04-17 14:38:45,388] Trial 0 finished with value: 0.7784578545189774 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7784578545189774.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:38:49,579] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8776371308016878
Fold 5 C-index: 0.7323943661971831
[I 2024-04-17 14:39:34,642] Trial 16 finished with value: 0.7903520467896392 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 95, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.884183184043622, 'min_weight_fraction_leaf': 0.10581506507448754}. Best is trial 3 with value: 0.802961090796227.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:39:35,925] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 4, 'n_estimators': 226, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.37217752523657055, 'min_weight_fraction_leaf': 0.1994767874105

Fold 4 C-index: 0.8860759493670886
Fold 5 C-index: 0.7746478873239436
[I 2024-04-17 14:39:54,800] Trial 31 finished with value: 0.8023701302106281 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 10, 'max_depth': 3, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.9175308644810205, 'min_weight_fraction_leaf': 0.1088935662786611}. Best is trial 3 with value: 0.802961090796227.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8776371308016878
Fold 5 C-index: 0.7652582159624414
[I 2024-04-17 14:39:56,105] Trial 32 finished with value: 0.7950579336258078 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 6, 'min_samples_leaf': 11, 'max_depth': 3, 'n_estimators': 211, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.9467713628420997, 'min_weight_fraction_leaf': 0.08138167552522578}. Best is trial 3

...

Fold 1 IBS: 0.18973195855840097
Fold 2 IBS: 0.19901752012782084
Fold 3 IBS: 0.1553163395473765
Fold 4 IBS: 0.13719070606611738
Fold 5 IBS: 0.22779776404493374
[I 2024-04-17 14:51:28,202] Trial 30 finished with value: 0.18181085766892985 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 305, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8220971907615917, 'min_weight_fraction_leaf': 0.076444606879153}. Best is trial 28 with value: 0.18105604370316586.
Fold 1 IBS: 0.2164877455822723
Fold 2 IBS: 0.18827035681808446
Fold 3 IBS: 0.16565039781780372
Fold 4 IBS: 0.1417428766421557
Fold 5 IBS: 0.21575647924950747
[I 2024-04-17 14:51:53,724] Trial 31 finished with value: 0.18558157122196473 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 394, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.91

Fold 1 IBS: 0.21196016007242774
Fold 2 IBS: 0.21737810285668177
Fold 3 IBS: 0.20221519268515586
Fold 4 IBS: 0.21985834792032022
Fold 5 IBS: 0.21707100228614096
[I 2024-04-17 14:54:16,677] Trial 45 finished with value: 0.21369656116414532 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 11, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 226, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.5998350649536632, 'min_weight_fraction_leaf': 0.05464331917294846}. Best is trial 33 with value: 0.18058942699185546.
Fold 1 IBS: 0.18076053104236317
Fold 2 IBS: 0.2172100382529918
Fold 3 IBS: 0.15472808053026357
Fold 4 IBS: 0.14511587127121353
Fold 5 IBS: 0.2339666086051307
[I 2024-04-17 14:54:22,837] Trial 46 finished with value: 0.18635622594039253 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 185, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.840

Fold 1 IBS: 0.1940353275621495
Fold 2 IBS: 0.19268012360689624
Fold 3 IBS: 0.18110607070067974
Fold 4 IBS: 0.18845548212963192
Fold 5 IBS: 0.2126254989546211
[I 2024-04-17 14:57:00,062] Trial 60 finished with value: 0.1937805005907957 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 12, 'max_depth': 12, 'n_estimators': 305, 'oob_score': True, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.8575527275336559, 'min_weight_fraction_leaf': 0.18982981109623043}. Best is trial 33 with value: 0.18058942699185546.
Fold 1 IBS: 0.1881809892682945
Fold 2 IBS: 0.19390719781855054
Fold 3 IBS: 0.1588739037458674
Fold 4 IBS: 0.1367801254919651
Fold 5 IBS: 0.2266847482237021
[I 2024-04-17 14:57:13,286] Trial 61 finished with value: 0.18088539290967592 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 19, 'n_estimators': 362, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7615

Fold 1 IBS: 0.20438643925191893
Fold 2 IBS: 0.1844312624795873
Fold 3 IBS: 0.17002168747641047
Fold 4 IBS: 0.14791525733420058
Fold 5 IBS: 0.21362624602582111
[I 2024-04-17 14:59:30,058] Trial 75 finished with value: 0.18407617851358768 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 11, 'n_estimators': 296, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.6651148142523113, 'min_weight_fraction_leaf': 0.0005666053087619444}. Best is trial 74 with value: 0.18029547957640585.
Fold 1 IBS: 0.19306514020322818
Fold 2 IBS: 0.1842925536567217
Fold 3 IBS: 0.16281493341095712
Fold 4 IBS: 0.15026196851923793
Fold 5 IBS: 0.20907571723485416
[I 2024-04-17 14:59:37,716] Trial 76 finished with value: 0.1799020626049998 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 273, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.6121

Fold 1 IBS: 0.19250257566167933
Fold 2 IBS: 0.18261531719008467
Fold 3 IBS: 0.16180974496548703
Fold 4 IBS: 0.14890218044290915
Fold 5 IBS: 0.2130135609453814
[I 2024-04-17 15:01:27,735] Trial 90 finished with value: 0.1797686758411083 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 202, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.5935532822743834, 'min_weight_fraction_leaf': 0.007362099342399538}. Best is trial 88 with value: 0.1785581840441121.
Fold 1 IBS: 0.19466401388017032
Fold 2 IBS: 0.18486336673650017
Fold 3 IBS: 0.16588446237479346
Fold 4 IBS: 0.1465134541216928
Fold 5 IBS: 0.21287370502296324
[I 2024-04-17 15:01:35,102] Trial 91 finished with value: 0.18095980042722398 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 197, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.5826023

In [62]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.886
train_ibs:  0.179


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=9, max_features=None, max_leaf_nodes=14,
                   max_samples=0.965941994806725, min_samples_split=7,
                   min_weight_fraction_leaf=0.017267854864073284,
                   n_estimators=336, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.662


ExtraSurvivalTrees(max_depth=9, max_features=0.1, max_leaf_nodes=20,
                   max_samples=0.5945245081220258, min_samples_leaf=4,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.006841645987133443,
                   n_estimators=279, oob_score=True, random_state=123,
                   warm_start=True)

IBS: 0.2


In [66]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 15:02:40,753] A new study created in memory with name: no-name-d71c9e2d-d4c3-4255-92c1-434ea5662e51


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:03:37,957] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:04:01,559] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:19:03,394] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7375077219334873.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:20:47,362] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:41:12,495] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.7577518089481858.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5654008438818565
Fold 5 C-index: 0.5821596244131455
[I 2024-04-17 15:43:02,130] Trial 26 finished with value: 0.5295120936590003 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632,

...

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-17 22:22:36,819] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.21552892877484067.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-17 22:22:55,446] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-17 22:26:50,642] Trial 66 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9998769833869746, 'learning_rate': 0.003967598379054899, 'dropout_rate': 0.17654958266634313, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.8981368148483696, 'min_weight_fraction_leaf': 0.03518122515344503, 'max_features': 'auto', 'min_impurity_decrease': 7.140027633149786e-05, 'validation_fraction': 0.6362228335648394, 'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 18, 'max_depth': 1}. Best is trial 41 with value: 0.21552892877484067.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-17 22:27:11,744] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8945386029541793, 'learning_rate': 0.01494075298406419, 'dropout_rate': 0.20

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-17 22:31:38,551] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7865219593038091, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.25073094184082545, 'n_estimators': 410, 'criterion': 'squared_error', 'ccp_alpha': 1.2846220433570537, 'min_weight_fraction_leaf': 0.32348748582358233, 'max_features': 1, 'min_impurity_decrease': 2.1196884133822008e-06, 'validation_fraction': 0.854746843935587, 'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 3}. Best is trial 75 with value: 0.20755403023595909.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-17 22:31:53,130] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.74450491092411, 'learning_rate': 0.08426314282635561,

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 22:36:38,399] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9047974654661481, 'learning_rate': 0.053489441535809173, 'dropout_rate': 0.15111205819257206, 'n_estimators': 351, 'criterion': 'squared_error', 'ccp_alpha': 0.8515398041309932, 'min_weight_fraction_leaf': 0.21790404115359022, 'max_features': 'auto', 'min_impurity_decrease': 2.7145979471070252e-06, 'validation_fraction': 0.8752408074269059, 'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 13, 'max_depth': 1}. Best is trial 75 with value: 0.20755403023595909.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-17 22:37:12,054] Trial 89 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8838515048108662, 'learning_rate': 0.011504505

Fold 3 IBS: 0.20030580413616672
Fold 4 IBS: 0.21827950204883104
Fold 5 IBS: 0.2160767264619057
[I 2024-04-17 22:42:33,320] Trial 99 finished with value: 0.21234381967629634 and parameters: {'subsample': 0.7984456569855936, 'learning_rate': 0.07921297533010697, 'dropout_rate': 0.36361374787631456, 'n_estimators': 498, 'criterion': 'squared_error', 'ccp_alpha': 0.004455045338579978, 'min_weight_fraction_leaf': 0.23447872920034368, 'max_features': 0.1, 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.4378698142708704, 'min_samples_split': 20, 'max_leaf_nodes': 20, 'min_samples_leaf': 15, 'max_depth': 5}. Best is trial 75 with value: 0.20755403023595909.


* Best trial for IBS: 
 FrozenTrial(number=75, state=TrialState.COMPLETE, values=[0.20755403023595909], datetime_start=datetime.datetime(2024, 4, 17, 22, 30, 26, 32885), datetime_complete=datetime.datetime(2024, 4, 17, 22, 30, 52, 843117), params={'subsample': 0.8646743646205938, 'learning_rate': 0.08843409287484

In [68]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.208


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0339977959383996,
                                 criterion='squared_error',
                                 dropout_rate=0.2075412325353082,
                                 learning_rate=0.010706280861824496,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=3.3602815261835675e-07,
                                 min_samples_leaf=13, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4472167339801619,
                                 n_estimators=445, random_state=123,
                                 subsample=0.9030031356045858,
                                 validation_fraction=0.9350158433232643)

C-index score: 0.618


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008810810992982535,
                                 criterion='squared_error',
                                 dropout_rate=0.1693875032679634,
                                 learning_rate=0.08843409287484169,
                                 max_features='auto', max_leaf_nodes=15,
                                 min_impurity_decrease=5.157320437854681e-06,
                                 min_samples_leaf=16, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29044798941528627,
                                 n_estimators=403, random_state=123,
                                 subsample=0.8646743646205938,
                                 validation_fraction=0.7489142536117352)

IBS: 0.213


In [72]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [74]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 22:42:45,720] A new study created in memory with name: no-name-350242cd-9097-4150-8ca0-4198759199d3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-17 22:42:48,493] Trial 0 finished with value: 0.6944636137409307 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6944636137409307.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-17 22:43:01,874] Trial 1 finished with value: 0.6934832215840679 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6944636137409307.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.6708860759493671
Fold 5 C-ind

Fold 1 C-index: 0.5757575757575758
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7464788732394366
[I 2024-04-17 22:45:24,540] Trial 19 finished with value: 0.7066289725991046 and parameters: {'subsample': 0.19816662555708264, 'dropout_rate': 0.34721836740782674, 'n_estimators': 400, 'learning_rate': 0.08909215792368923}. Best is trial 11 with value: 0.7242391503514646.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7511737089201878
[I 2024-04-17 22:45:32,109] Trial 20 finished with value: 0.7017096010027538 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.1874011856184571, 'n_estimators': 301, 'learning_rate': 0.06551866052750378}. Best is trial 11 with value: 0.7242391503514646.
Fold 1 C-index: 0.5844155844155844
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.679324894

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-17 22:48:26,120] Trial 38 finished with value: 0.6925028294272052 and parameters: {'subsample': 0.8179022915704841, 'dropout_rate': 0.46319782172467416, 'n_estimators': 475, 'learning_rate': 0.07288612365177065}. Best is trial 11 with value: 0.7242391503514646.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 22:48:29,914] Trial 39 finished with value: 0.6978708824167279 and parameters: {'subsample': 0.3183823327210974, 'dropout_rate': 0.22897359886270768, 'n_estimators': 145, 'learning_rate': 0.06741260221639067}. Best is trial 11 with value: 0.7242391503514646.
Fold 1 C-index: 0.5887445887445888
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.6877637130

...

Fold 3 IBS: 0.1756675093735378
Fold 4 IBS: 0.1919155264975053
Fold 5 IBS: 0.1849348585772
[I 2024-04-17 22:56:08,452] Trial 19 finished with value: 0.18708389706330752 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.18708389706330752.
Fold 1 IBS: 0.20469079919009717
Fold 2 IBS: 0.18161183535879627
Fold 3 IBS: 0.17748752476067053
Fold 4 IBS: 0.19624326076516657
Fold 5 IBS: 0.18897994630034004
[I 2024-04-17 22:56:09,723] Trial 20 finished with value: 0.18980267327501413 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.18708389706330752.
Fold 1 IBS: 0.2007782118192034
Fold 2 IBS: 0.18907326747918482
Fold 3 IBS: 0.18316466708155035
Fold 4 IBS: 0.20173113876039855
Fold 5 IBS: 0.19356091069665493
[I 2024-04-17 22:56:10,762] Trial 21 finishe

Fold 3 IBS: 0.1777438933250101
Fold 4 IBS: 0.19404071579693005
Fold 5 IBS: 0.17992947278624782
[I 2024-04-17 22:57:02,140] Trial 38 finished with value: 0.1856382621149739 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 82, 'learning_rate': 0.0684395247111694}. Best is trial 38 with value: 0.1856382621149739.
Fold 1 IBS: 0.2301128250403564
Fold 2 IBS: 0.17234930369004173
Fold 3 IBS: 0.1786611224369967
Fold 4 IBS: 0.2083426816157661
Fold 5 IBS: 0.18005256755486432
[I 2024-04-17 22:57:04,169] Trial 39 finished with value: 0.19390370006760504 and parameters: {'subsample': 0.1669627464432204, 'dropout_rate': 0.11471626893495929, 'n_estimators': 82, 'learning_rate': 0.06892741183938003}. Best is trial 38 with value: 0.1856382621149739.
Fold 1 IBS: 0.26127361158239537
Fold 2 IBS: 0.2702117097774595
Fold 3 IBS: 0.21315070084074397
Fold 4 IBS: 0.2757159479700989
Fold 5 IBS: 0.2205628674854048
[I 2024-04-17 22:57:06,860] Trial 40 finished 

Fold 3 IBS: 0.1939186074484014
Fold 4 IBS: 0.2132464949688479
Fold 5 IBS: 0.18111813126890497
[I 2024-04-17 22:57:50,860] Trial 57 finished with value: 0.2027049436643719 and parameters: {'subsample': 0.13165605060453595, 'dropout_rate': 0.23806025997264035, 'n_estimators': 81, 'learning_rate': 0.08927098403763303}. Best is trial 41 with value: 0.1826902124584475.
Fold 1 IBS: 0.21967163466408127
Fold 2 IBS: 0.1817845467825258
Fold 3 IBS: 0.1718116182769781
Fold 4 IBS: 0.21134882526508325
Fold 5 IBS: 0.178176570422679
[I 2024-04-17 22:57:52,267] Trial 58 finished with value: 0.1925586390822695 and parameters: {'subsample': 0.2849097248903497, 'dropout_rate': 0.17854033983790962, 'n_estimators': 52, 'learning_rate': 0.06792721960100616}. Best is trial 41 with value: 0.1826902124584475.
Fold 1 IBS: 0.20555111235130516
Fold 2 IBS: 0.2055417194065782
Fold 3 IBS: 0.19122632129312667
Fold 4 IBS: 0.2100252252563886
Fold 5 IBS: 0.20550842113648382
[I 2024-04-17 22:57:52,860] Trial 59 finished w

Fold 4 IBS: 0.24665592028691383
Fold 5 IBS: 0.19968789391964245
[I 2024-04-17 22:58:32,660] Trial 76 finished with value: 0.22329700875944378 and parameters: {'subsample': 0.16179134803259218, 'dropout_rate': 0.419783783711314, 'n_estimators': 109, 'learning_rate': 0.08167091626897967}. Best is trial 64 with value: 0.180321047381429.
Fold 1 IBS: 0.20604843080214338
Fold 2 IBS: 0.17946418794018407
Fold 3 IBS: 0.1748760996782727
Fold 4 IBS: 0.20466159091160982
Fold 5 IBS: 0.1858390531142425
[I 2024-04-17 22:58:33,584] Trial 77 finished with value: 0.1901778724892905 and parameters: {'subsample': 0.2243276930985941, 'dropout_rate': 0.12071886742003879, 'n_estimators': 29, 'learning_rate': 0.07730762793455846}. Best is trial 64 with value: 0.180321047381429.
Fold 1 IBS: 0.2578625406880957
Fold 2 IBS: 0.18836967898386145
Fold 3 IBS: 0.22090659208224933
Fold 4 IBS: 0.2174243607349852
Fold 5 IBS: 0.21571453745097394
[I 2024-04-17 22:58:36,825] Trial 78 finished with value: 0.22005554198803315

Fold 5 IBS: 0.17731417034945413
[I 2024-04-17 22:59:00,201] Trial 95 finished with value: 0.18617472119638018 and parameters: {'subsample': 0.1574638138642586, 'dropout_rate': 0.15912282102770056, 'n_estimators': 52, 'learning_rate': 0.09300288337363928}. Best is trial 64 with value: 0.180321047381429.
Fold 1 IBS: 0.20382017098407673
Fold 2 IBS: 0.16319103881220723
Fold 3 IBS: 0.17046677831779045
Fold 4 IBS: 0.20185197772656355
Fold 5 IBS: 0.18388128331588338
[I 2024-04-17 22:59:01,289] Trial 96 finished with value: 0.18464224983130428 and parameters: {'subsample': 0.11389291137795171, 'dropout_rate': 0.1513089117226007, 'n_estimators': 38, 'learning_rate': 0.09026893567244454}. Best is trial 64 with value: 0.180321047381429.
Fold 1 IBS: 0.20092420283967471
Fold 2 IBS: 0.18463472486015584
Fold 3 IBS: 0.17458222345731694
Fold 4 IBS: 0.19582668698716618
Fold 5 IBS: 0.1854345602840061
[I 2024-04-17 22:59:02,013] Trial 97 finished with value: 0.18828047968566394 and parameters: {'subsample

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.729
train_ibs:  0.18


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10064271403610489,
                                              learning_rate=0.09170799123141873,
                                              n_estimators=119,
                                              random_state=123,
                                              subsample=0.12850189742120152)

C-index score: 0.58


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.22493611916715012,
                                              learning_rate=0.09743269447662792,
                                              n_estimators=40, random_state=123,
                                              subsample=0.10180710042129003)

IBS: 0.237


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [80]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.886,1.0
Randomsurvivalforest,0.864,2.0
GradientBoosting,0.758,3.0
ComponentwiseGradientBoosting,0.729,4.0
CoxElastic,0.717,5.0
CoxLasso,0.706,6.0
CoxRidge,0.651,7.0


In [81]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.179,1.0
ComponentwiseGradientBoosting,0.180,2.0
Randomsurvivalforest,0.183,3.0
CoxElastic,0.195,4.0
CoxLasso,0.202,5.0
GradientBoosting,0.208,6.0
CoxRidge,0.217,7.0


In [82]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.662,1.0
Randomsurvivalforest,0.629,2.0
GradientBoosting,0.618,3.0
CoxLasso,0.597,4.5
CoxElastic,0.597,4.5
CoxRidge,0.592,6.0
ComponentwiseGradientBoosting,0.580,7.0


In [83]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ExtraSurvivalTrees,0.200,1.0
Randomsurvivalforest,0.205,2.0
GradientBoosting,0.213,3.0
CoxRidge,0.221,5.0
CoxLasso,0.221,5.0
CoxElastic,0.221,5.0
ComponentwiseGradientBoosting,0.237,7.0


In [84]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/robust/no_selection/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_robust_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [85]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-17
